# Language Model Probability Math — Theory Notebook

Interactive implementations covering the mathematical foundations of language model probability.

**Topics covered:**
1. Softmax function and numerical stability (log-sum-exp)
2. Cross-entropy loss and gradient derivation
3. Perplexity, entropy, KL divergence, bits-per-byte
4. Temperature scaling and distribution shaping
5. Decoding strategies: greedy, beam, top-k, top-p, min-p, typical
6. Scaling laws: Kaplan, Chinchilla, inference-optimal
7. DPO loss computation and gradient analysis
8. Calibration and ECE
9. Label smoothing
10. Full autoregressive generation pipeline

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available — skipping plots")

def print_vector(name, v, fmt=".4f"):
    """Pretty-print a named vector."""
    vals = ", ".join(f"{x:{fmt}}" for x in v)
    print(f"{name} = [{vals}]")

def print_matrix(name, M, fmt=".4f"):
    """Pretty-print a named matrix."""
    print(f"{name} =")
    for row in M:
        vals = "  ".join(f"{x:{fmt}}" for x in row)
        print(f"  [{vals}]")
    print()

## §1 Softmax Function and Numerical Stability

The softmax function converts raw logits $z \in \mathbb{R}^{|V|}$ to a probability distribution:

$$P(t_k) = \text{softmax}(z)_k = \frac{\exp(z_k)}{\sum_{v} \exp(z_v)}$$

**Problem**: For large logits, $\exp(z_v)$ overflows. The **log-sum-exp trick** fixes this:

$$\log \sum_v \exp(z_v) = a + \log \sum_v \exp(z_v - a), \quad a = \max(z)$$

In [ ]:
# === §1: Softmax and Log-Sum-Exp ===

def softmax_naive(z):
    """Naive softmax — numerically UNSTABLE for large logits."""
    e = np.exp(z)
    return e / e.sum()

def softmax_stable(z):
    """Numerically stable softmax using max-subtraction trick."""
    z_shifted = z - np.max(z)
    e = np.exp(z_shifted)
    return e / e.sum()

def log_softmax(z):
    """Numerically stable log-softmax (log-sum-exp trick)."""
    a = np.max(z)
    lse = a + np.log(np.sum(np.exp(z - a)))
    return z - lse

def log_sum_exp(z):
    """Numerically stable log-sum-exp."""
    a = np.max(z)
    return a + np.log(np.sum(np.exp(z - a)))

# --- Demo: Small logits (both methods agree) ---
z_small = np.array([2.0, 1.0, 0.5, -1.0])
print("=== Small Logits (z = [2.0, 1.0, 0.5, -1.0]) ===\n")
p_naive = softmax_naive(z_small)
p_stable = softmax_stable(z_small)
print_vector("softmax_naive ", p_naive)
print_vector("softmax_stable", p_stable)
print(f"Match: {np.allclose(p_naive, p_stable)}")
print(f"Sum: {p_stable.sum():.10f}")

# --- Demo: Large logits (naive overflows) ---
z_large = np.array([1000.0, 999.0, 998.0, 990.0])
print(f"\n=== Large Logits (z = [1000, 999, 998, 990]) ===\n")
try:
    p_naive_large = softmax_naive(z_large)
    print_vector("softmax_naive ", p_naive_large)
except Exception as e:
    print(f"softmax_naive: OVERFLOW — {e}")

# Check if we get inf/nan
with np.errstate(over='ignore', invalid='ignore'):
    p_naive_raw = softmax_naive(z_large)
    print(f"softmax_naive  produces nan: {np.any(np.isnan(p_naive_raw))}")

p_stable_large = softmax_stable(z_large)
print_vector("softmax_stable", p_stable_large)
print(f"Sum: {p_stable_large.sum():.10f}")

# --- Log-softmax ---
print(f"\n=== Log-Softmax ===\n")
ls = log_softmax(z_small)
print_vector("log_softmax(z_small)", ls)
print_vector("log(softmax(z_small))", np.log(p_stable))
print(f"Match: {np.allclose(ls, np.log(p_stable))}")

# --- Properties verification ---
print(f"\n=== Softmax Properties ===\n")
p = softmax_stable(z_small)
print(f"1. All non-negative: {np.all(p >= 0)}")
print(f"2. Sum to 1: {np.isclose(p.sum(), 1.0)}")
print(f"3. Monotone: argmax(z) = {np.argmax(z_small)}, argmax(p) = {np.argmax(p)}")
print(f"4. Translation invariant: softmax(z) == softmax(z + c)?")
p_shifted = softmax_stable(z_small + 100)
print(f"   {np.allclose(p, p_shifted)} (shift by +100)")

## §2 Cross-Entropy Loss and Gradient

The cross-entropy loss for a single token with logits $z$ and true token index $t^*$:

$$\mathcal{L} = -\log \text{softmax}(z)_{t^*} = -z_{t^*} + \log \sum_v \exp(z_v)$$

The gradient has the elegant form: $\frac{\partial \mathcal{L}}{\partial z_v} = \hat{p}_v - y_v$ (predicted minus true).

In [ ]:
# === §2: Cross-Entropy Loss and Gradient ===

def cross_entropy_loss(logits, target_idx):
    """
    Compute cross-entropy loss for a single token.
    L = -log softmax(z)_{target} = -z_{target} + log_sum_exp(z)
    """
    lse = log_sum_exp(logits)
    return -logits[target_idx] + lse

def cross_entropy_gradient(logits, target_idx):
    """
    Gradient of CE loss w.r.t. logits: ∂L/∂z_v = softmax(z)_v - 1[v == target]
    """
    p = softmax_stable(logits)
    grad = p.copy()
    grad[target_idx] -= 1.0
    return grad

# --- Worked example ---
z = np.array([2.0, 1.0, 0.5, -1.0])
target = 0  # correct token is index 0
vocab = ["the", "cat", "sat", "on"]

print("=== Cross-Entropy Loss: Worked Example ===\n")
print(f"Logits z = {z}")
print(f"True token: index {target} ('{vocab[target]}')")

# Compute loss two ways
loss_stable = cross_entropy_loss(z, target)
loss_direct = -np.log(softmax_stable(z)[target])
print(f"\nLoss (log-sum-exp): {loss_stable:.6f}")
print(f"Loss (-log softmax): {loss_direct:.6f}")
print(f"Match: {np.isclose(loss_stable, loss_direct)}")

# Compute probabilities
p = softmax_stable(z)
print(f"\n--- Predicted probabilities ---")
for i, (tok, prob, logit) in enumerate(zip(vocab, p, z)):
    marker = " ◄ target" if i == target else ""
    print(f"  P('{tok}') = {prob:.4f}  (logit = {logit:+.1f}){marker}")

# Compute gradient
grad = cross_entropy_gradient(z, target)
y = np.zeros(len(z))
y[target] = 1.0

print(f"\n--- Gradient ∂L/∂z (predicted - true) ---")
print(f"{'Token':<8} {'p̂':<10} {'y':<8} {'∂L/∂z':<12} {'Direction'}")
print("-" * 50)
for i, (tok, pi, yi, gi) in enumerate(zip(vocab, p, y, grad)):
    direction = "push DOWN ↓" if gi > 0 else "push UP ↑"
    print(f"{tok:<8} {pi:<10.4f} {yi:<8.1f} {gi:<+12.4f} {direction}")

print(f"\nSum of gradients: {grad.sum():.10f} (should be ≈0)")

# Numerical gradient check
print(f"\n--- Numerical Gradient Verification ---")
eps = 1e-5
for i in range(len(z)):
    z_plus = z.copy(); z_plus[i] += eps
    z_minus = z.copy(); z_minus[i] -= eps
    numerical_grad = (cross_entropy_loss(z_plus, target) - cross_entropy_loss(z_minus, target)) / (2 * eps)
    match = np.isclose(grad[i], numerical_grad, atol=1e-6)
    print(f"  ∂L/∂z_{i}: analytical={grad[i]:+.6f}, numerical={numerical_grad:+.6f} {'✓' if match else '✗'}")

## §3 Entropy, Cross-Entropy, KL Divergence, and Perplexity

The key information-theoretic quantities:

- **Entropy**: $H(P) = -\sum_t P(t) \log_2 P(t)$ — average surprise
- **Cross-entropy**: $H(P, Q) = -\sum_t P(t) \log_2 Q(t)$ — surprise using wrong model
- **KL divergence**: $D_{KL}(P \| Q) = H(P, Q) - H(P)$ — extra cost of using Q instead of P
- **Perplexity**: $\text{PPL} = 2^{H(P, Q)}$ — effective branching factor

In [ ]:
# === §3: Information-Theoretic Quantities ===

def entropy(p, base=2):
    """Shannon entropy H(P) in bits (base=2) or nats (base=e)."""
    p = p[p > 0]  # ignore zero entries
    return -np.sum(p * np.log(p)) / np.log(base)

def cross_entropy_dist(p, q, base=2):
    """Cross-entropy H(P, Q) = -sum P(t) log Q(t)."""
    mask = p > 0
    return -np.sum(p[mask] * np.log(q[mask])) / np.log(base)

def kl_divergence(p, q, base=2):
    """KL divergence D_KL(P || Q)."""
    mask = p > 0
    return np.sum(p[mask] * np.log(p[mask] / q[mask])) / np.log(base)

def perplexity_from_logprobs(log_probs):
    """Compute perplexity from per-token log-probabilities (nats)."""
    avg_nll = -np.mean(log_probs)
    return np.exp(avg_nll)

# --- Demo 1: Entropy of various distributions ---
V = 8  # small vocabulary for illustration
print("=== Entropy of Different Distributions ===\n")

dists = {
    "Uniform": np.ones(V) / V,
    "Peaked (one dominant)": np.array([0.7, 0.1, 0.05, 0.05, 0.04, 0.03, 0.02, 0.01]),
    "Very peaked": np.array([0.95, 0.01, 0.01, 0.01, 0.005, 0.005, 0.005, 0.005]),
    "Point mass": np.array([1.0, 0, 0, 0, 0, 0, 0, 0]),
}

print(f"{'Distribution':<25} {'H(P) bits':<12} {'Max entrpy':<12} {'PPL':<10}")
print("-" * 60)
for name, p in dists.items():
    H = entropy(p)
    ppl = 2**H
    print(f"{name:<25} {H:<12.4f} {np.log2(V):<12.4f} {ppl:<10.4f}")

# --- Demo 2: Cross-entropy and KL divergence ---
print(f"\n=== Cross-Entropy and KL Divergence ===\n")
P = np.array([0.4, 0.3, 0.2, 0.1])  # "true" distribution
Q1 = np.array([0.35, 0.25, 0.25, 0.15])  # close model
Q2 = np.array([0.1, 0.1, 0.4, 0.4])  # poor model
Q3 = np.ones(4) / 4  # uniform model

print(f"True distribution P = {P}")
print(f"H(P) = {entropy(P):.4f} bits\n")

models = {"Q1 (close)": Q1, "Q2 (poor)": Q2, "Q3 (uniform)": Q3}
print(f"{'Model':<16} {'H(P,Q) bits':<14} {'D_KL(P||Q)':<14} {'PPL':<10}")
print("-" * 56)
for name, Q in models.items():
    H_PQ = cross_entropy_dist(P, Q)
    D_KL = kl_divergence(P, Q)
    ppl = 2**H_PQ
    print(f"{name:<16} {H_PQ:<14.4f} {D_KL:<14.4f} {ppl:<10.4f}")

print(f"\nNote: H(P,Q) = H(P) + D_KL(P||Q)")
for name, Q in models.items():
    H_P = entropy(P)
    D = kl_divergence(P, Q)
    H_PQ = cross_entropy_dist(P, Q)
    print(f"  {name}: {H_P:.4f} + {D:.4f} = {H_P + D:.4f} vs H(P,Q) = {H_PQ:.4f} ✓")

# --- Demo 3: Perplexity from log-probabilities ---
print(f"\n=== Perplexity from Log-Probabilities ===\n")
# Simulated log-probs for a 5-token sequence
log_probs_nats = np.array([-2.3, -1.1, -3.5, -0.8, -2.0])
ppl = perplexity_from_logprobs(log_probs_nats)
avg_nll = -np.mean(log_probs_nats)
bits_per_token = avg_nll / np.log(2)

print(f"Log-probs (nats): {log_probs_nats}")
print(f"Average NLL: {avg_nll:.4f} nats")
print(f"Bits per token: {bits_per_token:.4f}")
print(f"Perplexity: {ppl:.4f}")
print(f"Interpretation: model is choosing from ~{ppl:.0f} tokens on average")

# Model comparison table
print(f"\n=== Model Perplexity Comparison (Reference) ===\n")
print(f"{'Model':<25} {'PPL (WikiText-103)':<22} {'Effective branching'}")
print("-" * 60)
models_ppl = [
    ("Random (|V|=50K)", 50000),
    ("5-gram LM", 70),
    ("GPT-2 (1.5B)", 18),
    ("GPT-3 (175B)", 10),
    ("LLaMA 3 70B", 4),
    ("Perfect model", 1),
]
for name, ppl_val in models_ppl:
    print(f"{name:<25} {ppl_val:<22} ~{ppl_val} candidates/step")

## §4 Temperature Scaling

Divide logits by temperature $\tau$ before softmax:

$$P_\tau(t) = \frac{\exp(z_t / \tau)}{\sum_v \exp(z_v / \tau)}$$

- $\tau < 1$: sharpens distribution (more deterministic)
- $\tau = 1$: standard softmax
- $\tau > 1$: flattens distribution (more random)
- $\tau \to 0$: approaches argmax; $\tau \to \infty$: approaches uniform

In [ ]:
# === §4: Temperature Scaling ===

def softmax_temperature(z, tau):
    """Softmax with temperature scaling."""
    return softmax_stable(z / tau)

z = np.array([3.0, 1.0, 0.5, -1.0])
vocab = ["Paris", "London", "Berlin", "banana"]
temps = [0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 5.0]

print("=== Temperature Scaling: logits = [3.0, 1.0, 0.5, -1.0] ===\n")
print(f"{'τ':<6}", end="")
for tok in vocab:
    print(f"  P({tok})", end="")
print(f"  {'H (bits)':<10} {'PPL':<8}")
print("-" * 72)

for tau in temps:
    p = softmax_temperature(z, tau)
    H = entropy(p)
    ppl = 2**H
    print(f"{tau:<6.1f}", end="")
    for pi in p:
        print(f"  {pi:<8.4f}", end="")
    print(f"  {H:<10.4f} {ppl:<8.2f}")

# Visualisation
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Panel 1: Distribution at different temperatures
    ax = axes[0]
    x = np.arange(len(vocab))
    select_temps = [0.3, 0.7, 1.0, 2.0]
    width = 0.8 / len(select_temps)
    colors = plt.cm.coolwarm(np.linspace(0.2, 0.8, len(select_temps)))
    for j, tau in enumerate(select_temps):
        p = softmax_temperature(z, tau)
        ax.bar(x + j * width, p, width, label=f'τ={tau}', color=colors[j], alpha=0.8)
    ax.set_xticks(x + width * (len(select_temps)-1) / 2)
    ax.set_xticklabels(vocab)
    ax.set_ylabel('Probability')
    ax.set_title('Token probabilities at different τ')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 2: Entropy vs temperature
    ax = axes[1]
    tau_range = np.linspace(0.05, 5.0, 200)
    entropies = [entropy(softmax_temperature(z, tau)) for tau in tau_range]
    ax.plot(tau_range, entropies, 'b-', linewidth=2)
    ax.axhline(y=np.log2(len(z)), color='red', linestyle='--', alpha=0.5, label=f'Max entropy ({np.log2(len(z)):.2f})')
    ax.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5, label='τ=1')
    ax.set_xlabel('Temperature τ')
    ax.set_ylabel('Entropy (bits)')
    ax.set_title('Entropy vs Temperature')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: Max probability vs temperature
    ax = axes[2]
    max_probs = [np.max(softmax_temperature(z, tau)) for tau in tau_range]
    ax.plot(tau_range, max_probs, 'r-', linewidth=2)
    ax.axhline(y=1.0/len(z), color='blue', linestyle='--', alpha=0.5, label=f'Uniform ({1/len(z):.2f})')
    ax.set_xlabel('Temperature τ')
    ax.set_ylabel('P(argmax token)')
    ax.set_title('Confidence vs Temperature')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('temperature_scaling.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: temperature_scaling.png")

## §5 Decoding Strategies

We implement all major decoding strategies and compare their behaviour on a synthetic vocabulary distribution.

In [ ]:
# === §5: Decoding Strategies ===

def greedy_decode(logits):
    """Greedy: always pick argmax."""
    return np.argmax(logits)

def top_k_sample(logits, k, temperature=1.0):
    """Top-k sampling: sample from k highest-probability tokens."""
    p = softmax_temperature(logits, temperature)
    top_k_idx = np.argsort(p)[-k:]
    mask = np.zeros_like(p)
    mask[top_k_idx] = p[top_k_idx]
    mask /= mask.sum()  # renormalise
    return np.random.choice(len(logits), p=mask), mask

def top_p_sample(logits, p_threshold, temperature=1.0):
    """Top-p (nucleus) sampling: sample from smallest set with cumprob >= p."""
    p = softmax_temperature(logits, temperature)
    sorted_idx = np.argsort(p)[::-1]  # descending
    cumsum = np.cumsum(p[sorted_idx])
    # Find smallest set with cumprob >= p_threshold
    cutoff = np.searchsorted(cumsum, p_threshold) + 1
    nucleus_idx = sorted_idx[:cutoff]
    mask = np.zeros_like(p)
    mask[nucleus_idx] = p[nucleus_idx]
    mask /= mask.sum()
    return np.random.choice(len(logits), p=mask), mask

def min_p_sample(logits, min_p_ratio, temperature=1.0):
    """Min-p sampling: keep tokens with P >= min_p_ratio * max(P)."""
    p = softmax_temperature(logits, temperature)
    threshold = min_p_ratio * np.max(p)
    mask = np.where(p >= threshold, p, 0.0)
    mask /= mask.sum()
    return np.random.choice(len(logits), p=mask), mask

def typical_sample(logits, delta, temperature=1.0):
    """Typical sampling: keep tokens with |I(t) - H(P)| <= delta."""
    p = softmax_temperature(logits, temperature)
    H = entropy(p, base=np.e)  # nats
    info = -np.log(p + 1e-30)  # self-information in nats
    typical_mask = np.abs(info - H) <= delta
    mask = np.where(typical_mask, p, 0.0)
    if mask.sum() == 0:
        mask = p  # fallback
    mask /= mask.sum()
    return np.random.choice(len(logits), p=mask), mask

# --- Demo with a realistic-ish distribution ---
np.random.seed(42)
V = 12
vocab = [f"tok_{i}" for i in range(V)]
# Simulated logits: a few high, most low
logits = np.array([4.0, 2.5, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -2.0, -3.0, -4.0])

p_orig = softmax_stable(logits)
print("=== Original Distribution ===\n")
print(f"{'Token':<8} {'Logit':<8} {'P(t)':<10} {'CumP':<10} {'I(t) bits':<10}")
print("-" * 50)
cumsum = 0
for i in range(V):
    cumsum += p_orig[i]
    info = -np.log2(p_orig[i]) if p_orig[i] > 0 else float('inf')
    print(f"{vocab[i]:<8} {logits[i]:<+8.1f} {p_orig[i]:<10.4f} {cumsum:<10.4f} {info:<10.4f}")

# --- Apply each strategy ---
print(f"\n=== Decoding Strategy Comparison ===\n")
strategies = {}

# Greedy
g = greedy_decode(logits)
mask_g = np.zeros(V); mask_g[g] = 1.0
strategies["Greedy"] = mask_g

# Top-k (k=3)
_, mask_k3 = top_k_sample(logits, k=3)
strategies["Top-k (k=3)"] = mask_k3

# Top-k (k=6)
_, mask_k6 = top_k_sample(logits, k=6)
strategies["Top-k (k=6)"] = mask_k6

# Top-p (p=0.9)
_, mask_p9 = top_p_sample(logits, p_threshold=0.9)
strategies["Top-p (p=0.9)"] = mask_p9

# Top-p (p=0.5)
_, mask_p5 = top_p_sample(logits, p_threshold=0.5)
strategies["Top-p (p=0.5)"] = mask_p5

# Min-p (ratio=0.1)
_, mask_mp = min_p_sample(logits, min_p_ratio=0.1)
strategies["Min-p (r=0.1)"] = mask_mp

# Typical (delta=1.0)
_, mask_typ = typical_sample(logits, delta=1.0)
strategies["Typical (δ=1)"] = mask_typ

# Print comparison
print(f"{'Strategy':<20} {'#tokens':<10} {'H (bits)':<10} {'Top token P':<12}")
print("-" * 55)
for name, mask in strategies.items():
    n_tokens = np.sum(mask > 0)
    H = entropy(mask) if n_tokens > 1 else 0
    top_p = np.max(mask)
    print(f"{name:<20} {n_tokens:<10.0f} {H:<10.4f} {top_p:<12.4f}")

# Visualisation
if HAS_MPL:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes_flat = axes.flatten()

    # Plot original distribution
    ax = axes_flat[0]
    ax.bar(range(V), p_orig, color='steelblue', alpha=0.8)
    ax.set_title('Original P(t)', fontsize=10)
    ax.set_ylabel('Probability')
    ax.set_xticks(range(V))
    ax.set_xticklabels([f't{i}' for i in range(V)], fontsize=7, rotation=45)
    ax.grid(True, alpha=0.3)

    # Plot each strategy
    for idx, (name, mask) in enumerate(strategies.items()):
        ax = axes_flat[idx + 1]
        colors = ['coral' if m > 0 else 'lightgray' for m in mask]
        ax.bar(range(V), mask, color=colors, alpha=0.8)
        ax.set_title(name, fontsize=10)
        ax.set_xticks(range(V))
        ax.set_xticklabels([f't{i}' for i in range(V)], fontsize=7, rotation=45)
        ax.grid(True, alpha=0.3)
        n_active = int(np.sum(mask > 0))
        ax.text(0.95, 0.95, f'n={n_active}', transform=ax.transAxes,
                ha='right', va='top', fontsize=9, fontweight='bold')

    plt.suptitle('Decoding Strategy Comparison', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('decoding_strategies.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: decoding_strategies.png")

## §6 Beam Search

Maintain $K$ candidate sequences (beams) and expand each by all vocabulary tokens at each step. Keep the top-$K$ by cumulative log-probability with length penalty.

In [ ]:
# === §6: Beam Search ===

def beam_search(log_prob_fn, V, K, max_len, length_penalty=0.7, bos=0, eos=None):
    """
    Simple beam search over a simulated LM.
    
    log_prob_fn(sequence) -> log_probs over V tokens
    Returns top-K completed sequences with scores.
    """
    # Each beam: (sequence, cumulative_log_prob)
    beams = [([bos], 0.0)]
    completed = []
    
    for step in range(max_len):
        candidates = []
        for seq, score in beams:
            log_probs = log_prob_fn(seq)
            for v in range(V):
                new_seq = seq + [v]
                new_score = score + log_probs[v]
                if eos is not None and v == eos:
                    completed.append((new_seq, new_score))
                else:
                    candidates.append((new_seq, new_score))
        
        # Keep top-K by score (with length penalty)
        candidates.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        beams = candidates[:K]
    
    # Add incomplete beams to completed
    completed.extend(beams)
    
    # Sort by length-normalised score
    completed.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
    return completed[:K]

# --- Simulated LM with position-dependent distributions ---
V = 5
token_names = ["the", "cat", "sat", "on", "mat"]

# Define a simple positional LM: P(token | position)
lm_table = {
    1: np.log(np.array([0.5, 0.15, 0.05, 0.2, 0.1])),   # after BOS
    2: np.log(np.array([0.05, 0.6, 0.1, 0.05, 0.2])),   # position 2
    3: np.log(np.array([0.05, 0.1, 0.6, 0.05, 0.2])),   # position 3
    4: np.log(np.array([0.1, 0.05, 0.05, 0.6, 0.2])),   # position 4
    5: np.log(np.array([0.1, 0.1, 0.1, 0.1, 0.6])),     # position 5
}

def simple_lm(seq):
    pos = len(seq)
    if pos in lm_table:
        return lm_table[pos]
    return np.log(np.ones(V) / V)  # uniform for later positions

print("=== Beam Search Demo ===\n")
print(f"Vocabulary: {token_names}\n")
print("LM probabilities per position:")
for pos in range(1, 6):
    p = np.exp(lm_table[pos])
    print(f"  pos {pos}: {dict(zip(token_names, [f'{x:.2f}' for x in p]))}")

# Run beam search with different K
for K in [1, 3, 5]:
    results = beam_search(simple_lm, V, K=K, max_len=5, length_penalty=0.7)
    print(f"\n--- Beam search (K={K}) ---")
    for rank, (seq, score) in enumerate(results):
        tokens = [token_names[t] for t in seq]
        norm_score = score / (len(seq) ** 0.7)
        print(f"  #{rank+1}: {' '.join(tokens[1:])} | log P = {score:.3f} | normalised = {norm_score:.3f}")

# Compare greedy vs beam
print(f"\nGreedy sequence:", end=" ")
seq = [0]  # BOS
for step in range(5):
    lp = simple_lm(seq)
    best = np.argmax(lp)
    seq.append(best)
    print(token_names[best], end=" ")
greedy_score = sum(simple_lm(seq[:i+1])[seq[i+1]] for i in range(5))
print(f"\n  log P(greedy) = {greedy_score:.3f}")

beam_results = beam_search(simple_lm, V, K=5, max_len=5, length_penalty=0.7)
beam_score = beam_results[0][1]
print(f"  log P(beam-5) = {beam_score:.3f}")
print(f"  Beam search found {'better' if beam_score > greedy_score else 'same'} sequence")

## §7 Scaling Laws

Neural scaling laws predict loss as a function of model parameters $N$, data $D$, and compute $C$:

- **Kaplan**: $L(N) = (N_c/N)^{0.076}$, emphasises parameters over data
- **Chinchilla**: $N_{opt} \propto C^{0.5}$, $D_{opt} \propto C^{0.5}$, ~20 tokens/param
- **Inference-optimal**: Over-train small models for cheaper deployment

In [ ]:
# === §7: Scaling Laws ===

# --- Kaplan scaling law: L(N) = (Nc/N)^alpha ---
alpha_N = 0.076
alpha_D = 0.095
alpha_C = 0.050

# Fit Nc from GPT-3 reference point: 175B params → ~1.73 nats (WikiText-103)
L_ref = 1.73
N_ref = 175e9
Nc = N_ref * (L_ref ** (1 / alpha_N))

print("=== Kaplan Scaling Laws ===\n")
print(f"Reference: GPT-3 175B → L = {L_ref} nats")
print(f"Fitted Nc = {Nc:.2e}\n")

model_sizes = [125e6, 350e6, 1.3e9, 6.7e9, 13e9, 70e9, 175e9, 540e9, 1e12]
model_names = ["125M", "350M", "1.3B", "6.7B", "13B", "70B", "175B", "540B", "1T"]

print(f"{'Model size':<12} {'N (params)':<14} {'L(N) nats':<12} {'PPL':<10} {'Δ from 175B'}")
print("-" * 60)
for name, N in zip(model_names, model_sizes):
    L = (Nc / N) ** alpha_N
    ppl = np.exp(L)
    delta = (L / L_ref - 1) * 100
    print(f"{name:<12} {N:<14.2e} {L:<12.4f} {ppl:<10.2f} {delta:+.1f}%")

# --- Chinchilla optimal allocation ---
print(f"\n=== Chinchilla: Optimal Tokens per Parameter ===\n")
chinchilla_ratio = 20  # tokens per parameter

print(f"{'Model':<12} {'Params':<14} {'Chinchilla D':<14} {'Actual D':<14} {'Actual ratio':<14}")
print("-" * 70)
real_models = [
    ("GPT-3", 175e9, 300e9),
    ("Chinchilla", 70e9, 1.4e12),
    ("LLaMA 2 70B", 70e9, 2e12),
    ("LLaMA 3 8B", 8e9, 15e12),
    ("LLaMA 3 70B", 70e9, 15e12),
]
for name, N, D_actual in real_models:
    D_optimal = chinchilla_ratio * N
    ratio = D_actual / N
    print(f"{name:<12} {N/1e9:.0f}B{'':<9} {D_optimal/1e12:.1f}T{'':<10} {D_actual/1e12:.1f}T{'':<10} {ratio:.0f}:1")

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # Panel 1: L(N) power law
    ax = axes[0]
    N_range = np.logspace(7, 13, 200)
    L_range = (Nc / N_range) ** alpha_N
    ax.semilogx(N_range, L_range, 'b-', linewidth=2, label=f'L(N) = (Nc/N)^{alpha_N}')
    # Plot real model reference points
    for name, N in zip(model_names, model_sizes):
        L = (Nc / N) ** alpha_N
        ax.plot(N, L, 'ro', markersize=5)
        if name in ["125M", "1.3B", "13B", "175B"]:
            ax.annotate(name, (N, L), fontsize=7, xytext=(5, 5), textcoords='offset points')
    ax.set_xlabel('Parameters N')
    ax.set_ylabel('Loss (nats)')
    ax.set_title('Kaplan: Loss vs Parameters')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    # Panel 2: Chinchilla compute-optimal frontier
    ax = axes[1]
    C_range = np.logspace(18, 26, 200)  # FLOPs
    N_chinchilla = np.sqrt(C_range / (6 * chinchilla_ratio))  # C ≈ 6ND
    D_chinchilla = chinchilla_ratio * N_chinchilla
    ax.loglog(C_range, N_chinchilla, 'b-', linewidth=2, label='Optimal N')
    ax.loglog(C_range, D_chinchilla, 'r--', linewidth=2, label='Optimal D')
    ax.set_xlabel('Compute C (FLOPs)')
    ax.set_ylabel('N (params) / D (tokens)')
    ax.set_title('Chinchilla: Optimal Allocation')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: Over-training ratio in real models
    ax = axes[2]
    names = [m[0] for m in real_models]
    ratios = [m[2]/m[1] for m in real_models]
    colors = ['red' if r < 20 else ('green' if r < 50 else 'blue') for r in ratios]
    bars = ax.barh(names, ratios, color=colors, alpha=0.7, edgecolor='black')
    ax.axvline(x=20, color='red', linestyle='--', label='Chinchilla optimal (20:1)')
    ax.set_xlabel('Tokens per parameter')
    ax.set_title('Training Data Ratio')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    # Log scale for x
    ax.set_xscale('log')

    plt.tight_layout()
    plt.savefig('scaling_laws.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: scaling_laws.png")

## §8 DPO Loss and Alignment

**Bradley-Terry preference model:**
$$P(\text{winner} = y_w \mid y_w, y_l) = \sigma\bigl(r(y_w) - r(y_l)\bigr)$$

**DPO reparametrisation** — reward is implicit in log-probability ratios:
$$r(y) = \beta \log\frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \text{const}$$

**DPO loss** (closed-form, no reward model needed):
$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\Bigl(\beta\bigl[\log\frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \log\frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\bigr]\Bigr)$$

We implement DPO loss, compute gradients, and compare with RLHF/PPO.

In [ ]:
# === §8: DPO Loss and Alignment ===
np.random.seed(42)

def sigmoid(x):
    """Numerically stable sigmoid."""
    return np.where(x >= 0,
                    1 / (1 + np.exp(-x)),
                    np.exp(x) / (1 + np.exp(x)))

def dpo_loss(logp_w_theta, logp_l_theta, logp_w_ref, logp_l_ref, beta=0.1):
    """
    DPO loss for a single preference pair.
    L = -log σ(β * [(logπ_θ(y_w) - logπ_ref(y_w)) - (logπ_θ(y_l) - logπ_ref(y_l))])
    """
    log_ratio_w = logp_w_theta - logp_w_ref  # implicit reward for winner
    log_ratio_l = logp_l_theta - logp_l_ref  # implicit reward for loser
    margin = beta * (log_ratio_w - log_ratio_l)
    loss = -np.log(sigmoid(margin) + 1e-10)
    return loss, margin

def dpo_gradient(logp_w_theta, logp_l_theta, logp_w_ref, logp_l_ref, beta=0.1):
    """
    DPO gradient: ∂L/∂logπ_θ signals.
    grad_w = -β * σ(-margin)  (push winner probability UP)
    grad_l = +β * σ(-margin)  (push loser probability DOWN)
    """
    log_ratio_w = logp_w_theta - logp_w_ref
    log_ratio_l = logp_l_theta - logp_l_ref
    margin = beta * (log_ratio_w - log_ratio_l)
    weight = sigmoid(-margin)  # how "wrong" the model currently is
    grad_w = -beta * weight    # negative → increase log-prob of winner
    grad_l = +beta * weight    # positive → decrease log-prob of loser
    return grad_w, grad_l, weight

# --- Worked example ---
print("=== DPO Worked Example ===\n")
print("Prompt: 'Explain quantum computing in simple terms'\n")

# Simulate log-probabilities for winner/loser responses
# Under reference model (SFT checkpoint)
logp_w_ref = -3.2   # winner response log-prob under ref
logp_l_ref = -2.8   # loser response log-prob under ref (loser was more likely under ref!)

# Under current policy (being trained)
logp_w_theta = -2.5  # winner pushed up  
logp_l_theta = -3.1  # loser pushed down

beta_values = [0.05, 0.1, 0.2, 0.5, 1.0]

print(f"  logπ_ref(y_w) = {logp_w_ref},  logπ_ref(y_l) = {logp_l_ref}")
print(f"  logπ_θ(y_w)   = {logp_w_theta},  logπ_θ(y_l)   = {logp_l_theta}")
print()

print(f"{'β':<8} {'margin':<10} {'σ(margin)':<12} {'Loss':<10} {'grad_w':<10} {'grad_l':<10} {'weight':<10}")
print("-" * 72)
for beta in beta_values:
    loss, margin = dpo_loss(logp_w_theta, logp_l_theta, logp_w_ref, logp_l_ref, beta)
    grad_w, grad_l, weight = dpo_gradient(logp_w_theta, logp_l_theta, logp_w_ref, logp_l_ref, beta)
    sig_margin = sigmoid(margin)
    print(f"{beta:<8.2f} {margin:<10.4f} {sig_margin:<12.4f} {loss:<10.4f} {grad_w:<10.4f} {grad_l:<10.4f} {weight:<10.4f}")

# --- DPO training trajectory ---
print("\n=== DPO Training Trajectory ===\n")
print("Simulating DPO updates on a preference pair...\n")

logp_w = -3.0  # start: winner and loser have similar log-prob
logp_l = -2.9  # loser slightly preferred initially
logp_w_r = -3.0  # reference stays fixed
logp_l_r = -2.9
beta = 0.1
lr = 0.5  # learning rate (exaggerated for demo)

print(f"{'Step':<6} {'logπ(y_w)':<12} {'logπ(y_l)':<12} {'margin':<10} {'Loss':<10} {'P(w>l)':<10}")
print("-" * 62)
for step in range(15):
    loss, margin = dpo_loss(logp_w, logp_l, logp_w_r, logp_l_r, beta)
    grad_w, grad_l, weight = dpo_gradient(logp_w, logp_l, logp_w_r, logp_l_r, beta)
    p_correct = sigmoid(margin)
    print(f"{step:<6} {logp_w:<12.4f} {logp_l:<12.4f} {margin:<10.4f} {loss:<10.4f} {p_correct:<10.4f}")
    # Gradient update
    logp_w -= lr * grad_w  # increase winner log-prob
    logp_l -= lr * grad_l  # decrease loser log-prob

# --- DPO vs PPO comparison ---
print("\n=== DPO vs PPO/RLHF Comparison ===\n")
comparison = [
    ("Component", "PPO (RLHF)", "DPO"),
    ("Reward model", "Separate NN trained on preferences", "Implicit (log-ratio)"),
    ("Policy optim", "RL (clipped surrogate)", "Supervised (cross-entropy-like)"),
    ("KL penalty", "Explicit β·D_KL in reward", "Implicit via ref model ratio"),
    ("Stability", "Sensitive to hyperparams", "Stable, fewer hyperparams"),
    ("Compute", "4 models (policy, ref, reward, critic)", "2 models (policy, ref)"),
    ("Memory", "~4× model size", "~2× model size"),
    ("β controls", "KL regularisation strength", "Deviation from reference"),
]
for row in comparison:
    if row == comparison[0]:
        print(f"  {row[0]:<16} {row[1]:<38} {row[2]}")
        print("  " + "-" * 80)
    else:
        print(f"  {row[0]:<16} {row[1]:<38} {row[2]}")

# Visualization
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))

    # Panel 1: DPO loss as function of margin
    ax = axes[0]
    margins = np.linspace(-5, 5, 200)
    for beta in [0.05, 0.1, 0.5, 1.0]:
        losses = -np.log(sigmoid(beta * margins) + 1e-10)
        ax.plot(margins, losses, linewidth=2, label=f'β={beta}')
    ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Margin (log_ratio_w - log_ratio_l)')
    ax.set_ylabel('DPO Loss')
    ax.set_title('DPO Loss vs Margin')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 5)

    # Panel 2: Training trajectory
    ax = axes[1]
    logp_w, logp_l = -3.0, -2.9
    beta, lr = 0.1, 0.5
    w_traj, l_traj, loss_traj = [logp_w], [logp_l], []
    for _ in range(20):
        loss, _ = dpo_loss(logp_w, logp_l, logp_w_r, logp_l_r, beta)
        grad_w, grad_l, _ = dpo_gradient(logp_w, logp_l, logp_w_r, logp_l_r, beta)
        loss_traj.append(loss)
        logp_w -= lr * grad_w
        logp_l -= lr * grad_l
        w_traj.append(logp_w)
        l_traj.append(logp_l)
    ax.plot(w_traj, 'g-o', markersize=3, label='logπ(y_winner)')
    ax.plot(l_traj, 'r-o', markersize=3, label='logπ(y_loser)')
    ax.set_xlabel('Step')
    ax.set_ylabel('Log-probability')
    ax.set_title('DPO Training Trajectory')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Panel 3: Gradient magnitude over training
    ax = axes[2]
    ax.plot(loss_traj, 'b-o', markersize=3)
    ax.set_xlabel('Step')
    ax.set_ylabel('DPO Loss')
    ax.set_title('Loss Convergence')
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('dpo_alignment.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: dpo_alignment.png")

## §9 Calibration and Expected Calibration Error

A model is **calibrated** if its confidence matches its accuracy:
$$P(\hat{y} = y \mid \text{conf} = p) = p$$

**Expected Calibration Error (ECE):**
$$\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{N} \bigl|\text{acc}(B_m) - \text{conf}(B_m)\bigr|$$

where $B_m$ are confidence bins, $\text{acc}(B_m)$ is accuracy in bin $m$, and $\text{conf}(B_m)$ is mean confidence.

**Temperature scaling** post-hoc recalibration:
$$\hat{p} = \text{softmax}(z / T^*), \quad T^* = \arg\min_T \text{NLL}_{\text{val}}(z/T)$$

In [ ]:
# === §9: Calibration and Expected Calibration Error ===
np.random.seed(42)

def compute_ece(confidences, accuracies, n_bins=10):
    """
    Expected Calibration Error.
    confidences: max softmax probability for each prediction
    accuracies: 1 if correct, 0 if wrong
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_data = []
    
    for i in range(n_bins):
        lo, hi = bin_boundaries[i], bin_boundaries[i + 1]
        mask = (confidences > lo) & (confidences <= hi)
        if mask.sum() == 0:
            bin_data.append((lo, hi, 0, 0, 0, 0))
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        bin_count = mask.sum()
        gap = abs(bin_acc - bin_conf)
        ece += (bin_count / len(confidences)) * gap
        bin_data.append((lo, hi, bin_count, bin_acc, bin_conf, gap))
    
    return ece, bin_data

def temperature_scale(logits, T):
    """Apply temperature scaling to logits."""
    scaled = logits / T
    return softmax_stable(scaled)

# --- Simulate an overconfident LLM ---
N = 500
np.random.seed(42)

# Overconfident model: high confidence but mediocre accuracy
true_labels = np.random.randint(0, 10, size=N)
# Generate logits: correct class gets moderate boost + noise
logits_all = np.random.randn(N, 10) * 0.5
for i in range(N):
    logits_all[i, true_labels[i]] += 2.0  # boost correct class
    # Add overconfidence: sharpen the distribution
    logits_all[i] *= 2.5  # makes softmax peaky → overconfident

# Compute predictions
probs_all = np.array([softmax_stable(z) for z in logits_all])
predicted_labels = np.argmax(probs_all, axis=1)
max_confs = np.max(probs_all, axis=1)
correct = (predicted_labels == true_labels).astype(float)

accuracy = correct.mean()
mean_confidence = max_confs.mean()

print("=== Calibration Analysis: Overconfident Model ===\n")
print(f"Overall accuracy:   {accuracy:.3f}")
print(f"Mean confidence:    {mean_confidence:.3f}")
print(f"Confidence gap:     {mean_confidence - accuracy:.3f} (overconfident!)")
print()

ece_before, bins_before = compute_ece(max_confs, correct, n_bins=10)
print(f"ECE (before calibration): {ece_before:.4f}\n")

print(f"{'Bin':<12} {'Count':<8} {'Acc':<8} {'Conf':<8} {'|Gap|':<8}")
print("-" * 44)
for lo, hi, count, acc, conf, gap in bins_before:
    if count > 0:
        print(f"({lo:.1f}, {hi:.1f}]{'':<3} {count:<8.0f} {acc:<8.3f} {conf:<8.3f} {gap:<8.3f}")

# --- Temperature scaling for recalibration ---
print("\n=== Temperature Scaling Recalibration ===\n")

# Find optimal temperature by minimising NLL on the data
best_T, best_nll = 1.0, float('inf')
for T_cand in np.arange(0.5, 5.0, 0.05):
    nll = 0.0
    for i in range(N):
        p = temperature_scale(logits_all[i], T_cand)
        nll -= np.log(p[true_labels[i]] + 1e-10)
    if nll < best_nll:
        best_nll = nll
        best_T = T_cand

print(f"Optimal temperature T* = {best_T:.2f}\n")

# Apply optimal temperature
probs_calibrated = np.array([temperature_scale(z, best_T) for z in logits_all])
max_confs_cal = np.max(probs_calibrated, axis=1)
# Predictions don't change (temperature preserves argmax)
ece_after, bins_after = compute_ece(max_confs_cal, correct, n_bins=10)

print(f"{'Metric':<30} {'Before':<12} {'After':<12}")
print("-" * 54)
print(f"{'ECE':<30} {ece_before:<12.4f} {ece_after:<12.4f}")
print(f"{'Mean confidence':<30} {mean_confidence:<12.3f} {max_confs_cal.mean():<12.3f}")
print(f"{'Accuracy (unchanged)':<30} {accuracy:<12.3f} {accuracy:<12.3f}")
print(f"{'Max confidence':<30} {max_confs.max():<12.3f} {max_confs_cal.max():<12.3f}")

# Visualization: Reliability diagram
if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    
    # Panel 1: Reliability diagram BEFORE
    ax = axes[0]
    bin_mids = [(lo + hi) / 2 for lo, hi, c, a, co, g in bins_before if c > 0]
    bin_accs = [a for _, _, c, a, _, _ in bins_before if c > 0]
    bin_confs_plot = [co for _, _, c, _, co, _ in bins_before if c > 0]
    ax.bar(bin_mids, bin_accs, width=0.08, alpha=0.6, color='steelblue', edgecolor='black', label='Accuracy')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect calibration')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'Before (ECE={ece_before:.3f})')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    
    # Panel 2: Reliability diagram AFTER
    ax = axes[1]
    bin_mids2 = [(lo + hi) / 2 for lo, hi, c, a, co, g in bins_after if c > 0]
    bin_accs2 = [a for _, _, c, a, _, _ in bins_after if c > 0]
    ax.bar(bin_mids2, bin_accs2, width=0.08, alpha=0.6, color='green', edgecolor='black', label='Accuracy')
    ax.plot([0, 1], [0, 1], 'r--', linewidth=2, label='Perfect calibration')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'After T*={best_T:.2f} (ECE={ece_after:.3f})')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    
    # Panel 3: Confidence distributions before/after
    ax = axes[2]
    ax.hist(max_confs, bins=30, alpha=0.5, color='red', label='Before', density=True)
    ax.hist(max_confs_cal, bins=30, alpha=0.5, color='green', label='After T*', density=True)
    ax.axvline(accuracy, color='black', linestyle='--', label=f'True acc={accuracy:.2f}')
    ax.set_xlabel('Max confidence')
    ax.set_ylabel('Density')
    ax.set_title('Confidence Distribution')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('calibration.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: calibration.png")

## §10 Label Smoothing

Standard one-hot target: $y = e_k$ (1 at position $k$, 0 elsewhere).

**Label smoothing** distributes $\epsilon$ probability mass uniformly:
$$y_{\text{smooth}}(i) = \begin{cases} 1 - \epsilon + \epsilon/V & \text{if } i = k \\ \epsilon / V & \text{otherwise}\end{cases}$$

**Effect on loss:** prevents the model from becoming infinitely confident:
$$\mathcal{L}_{\text{smooth}} = (1-\epsilon)\mathcal{L}_{\text{CE}} + \epsilon \cdot H(u, \hat{p})$$

where $H(u, \hat{p})$ is cross-entropy with the uniform distribution — a **confidence penalty**.

In [ ]:
# === §10: Label Smoothing ===

def label_smooth(target_idx, vocab_size, epsilon=0.1):
    """Create label-smoothed target distribution."""
    y = np.full(vocab_size, epsilon / vocab_size)
    y[target_idx] += (1 - epsilon)
    return y

def cross_entropy_with_smoothing(logits, target_idx, epsilon=0.0):
    """Cross-entropy loss with optional label smoothing."""
    V = len(logits)
    p = softmax_stable(logits)
    
    if epsilon == 0:
        return -np.log(p[target_idx] + 1e-10)
    else:
        y_smooth = label_smooth(target_idx, V, epsilon)
        return -np.sum(y_smooth * np.log(p + 1e-10))

# --- Demonstrate label smoothing ---
print("=== Label Smoothing ===\n")
V = 8
target = 2
vocab = ["the", "cat", "sat", "on", "a", "mat", "is", "big"]

print(f"Vocab size V = {V}, target = '{vocab[target]}' (index {target})\n")

epsilons = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
print(f"{'ε':<8} {'y[target]':<12} {'y[other]':<12} {'Sum(y)':<8}")
print("-" * 44)
for eps in epsilons:
    y = label_smooth(target, V, eps)
    print(f"{eps:<8.2f} {y[target]:<12.4f} {y[0]:<12.6f} {y.sum():<8.4f}")

# --- Effect on loss and gradient ---
print("\n=== Effect on Training ===\n")
print("  Logits z = [3.0, 1.0, 5.0, 0.5, 0.0, -1.0, 0.5, -0.5]")
logits = np.array([3.0, 1.0, 5.0, 0.5, 0.0, -1.0, 0.5, -0.5])
p = softmax_stable(logits)
print(f"  softmax(z) = [{', '.join(f'{x:.4f}' for x in p)}]")
print(f"  p[target='{vocab[target]}'] = {p[target]:.4f}\n")

print(f"{'ε':<8} {'Loss(ε)':<12} {'∂L/∂z[target]':<16} {'Max |∂L/∂z|':<14} {'Gradient entropy':<18}")
print("-" * 70)
for eps in epsilons:
    loss = cross_entropy_with_smoothing(logits, target, eps)
    
    # Compute gradient numerically
    grad = np.zeros(V)
    delta = 1e-5
    for j in range(V):
        logits_plus = logits.copy(); logits_plus[j] += delta
        logits_minus = logits.copy(); logits_minus[j] -= delta
        grad[j] = (cross_entropy_with_smoothing(logits_plus, target, eps) - 
                    cross_entropy_with_smoothing(logits_minus, target, eps)) / (2*delta)
    
    # Gradient entropy (more spread = more regularised)
    grad_mag = np.abs(grad)
    grad_dist = grad_mag / (grad_mag.sum() + 1e-10)
    grad_ent = -np.sum(grad_dist * np.log(grad_dist + 1e-10))
    
    print(f"{eps:<8.2f} {loss:<12.4f} {grad[target]:<16.4f} {np.max(np.abs(grad)):<14.4f} {grad_ent:<18.4f}")

# --- Analytical gradient comparison ---
print("\n=== Label Smoothing Gradient: Analytical ===\n")
print("With hard labels:  ∂L/∂z_i = p_i - 1[i=k]")
print("With smooth labels: ∂L/∂z_i = p_i - y_smooth_i")
print()

eps = 0.1
y_hard = np.zeros(V); y_hard[target] = 1.0
y_smooth = label_smooth(target, V, eps)

print(f"{'Token':<8} {'p_i':<10} {'y_hard':<10} {'y_smooth':<10} {'grad_hard':<12} {'grad_smooth':<12}")
print("-" * 64)
for i in range(V):
    g_hard = p[i] - y_hard[i]
    g_smooth = p[i] - y_smooth[i]
    print(f"{vocab[i]:<8} {p[i]:<10.4f} {y_hard[i]:<10.1f} {y_smooth[i]:<10.4f} {g_hard:<12.4f} {g_smooth:<12.4f}")

print(f"\nKey insight: smoothing pushes non-target gradients toward NEGATIVE")
print(f"  → model doesn't fully suppress non-target tokens")
print(f"  → prevents overconfidence, improves generalisation")

## §11 Full Autoregressive Generation Pipeline

Putting it all together: simulate the **complete autoregressive loop** of a language model.

**Pipeline:** embedding lookup → linear projection → softmax → sample → append → repeat

$$\text{for } t = 1, \ldots, T: \quad x_t \sim P_\theta(\cdot \mid x_{<t}) = \text{softmax}\bigl(W_{\text{head}} \cdot h_t / \tau\bigr)$$

We simulate a tiny LM with a learned transition matrix and show how different decoding strategies affect the generated text, probability, and perplexity of the output.

In [ ]:
# === §11: Full Autoregressive Generation Pipeline ===
np.random.seed(42)

# --- Define a tiny language model ---
vocab_tiny = ["<bos>", "the", "cat", "sat", "on", "a", "mat", "dog", "ran", "fast", "<eos>"]
V_tiny = len(vocab_tiny)
id2tok = {i: w for i, w in enumerate(vocab_tiny)}
tok2id = {w: i for i, w in enumerate(vocab_tiny)}

# Transition logits: simulate a "trained" LM via a transition matrix
# Each row = logits for P(next | current)
# Encodes: "the cat sat on a mat" and "the dog ran fast" as high-probability paths
transition_logits = np.random.randn(V_tiny, V_tiny) * 0.3  # base noise

# Strong transitions
strong_pairs = [
    ("<bos>", "the", 4.0),
    ("the", "cat", 3.5), ("the", "dog", 3.0), ("the", "mat", 1.5),
    ("cat", "sat", 4.0), ("cat", "ran", 1.0),
    ("sat", "on", 4.5),
    ("on", "a", 4.0), ("on", "the", 2.0),
    ("a", "mat", 3.5), ("a", "cat", 1.5), ("a", "dog", 1.5),
    ("mat", "<eos>", 4.0),
    ("dog", "ran", 4.0), ("dog", "sat", 1.5),
    ("ran", "fast", 4.0), ("ran", "on", 1.0),
    ("fast", "<eos>", 4.5),
]
for src, tgt, strength in strong_pairs:
    transition_logits[tok2id[src], tok2id[tgt]] = strength

def tiny_lm_logits(context):
    """Get next-token logits from our tiny LM (uses last token only = bigram)."""
    last_token_id = context[-1]
    return transition_logits[last_token_id].copy()

def generate_sequence(start_token, strategy, max_len=10, temperature=1.0, 
                      top_k=None, top_p=None, min_p_thresh=None):
    """Full autoregressive generation."""
    context = [tok2id[start_token]]
    log_probs = []
    
    for _ in range(max_len):
        logits = tiny_lm_logits(context)
        
        # Apply temperature
        logits_scaled = logits / temperature
        probs = softmax_stable(logits_scaled)
        
        if strategy == "greedy":
            next_id = np.argmax(probs)
        elif strategy == "sample":
            next_id = np.random.choice(V_tiny, p=probs)
        elif strategy == "top_k":
            top_idx = np.argsort(probs)[::-1][:top_k]
            top_probs = probs[top_idx]
            top_probs /= top_probs.sum()
            next_id = np.random.choice(top_idx, p=top_probs)
        elif strategy == "top_p":
            sorted_idx = np.argsort(probs)[::-1]
            cumsum = np.cumsum(probs[sorted_idx])
            cutoff = np.searchsorted(cumsum, top_p) + 1
            nucleus = sorted_idx[:cutoff]
            nuc_probs = probs[nucleus]
            nuc_probs /= nuc_probs.sum()
            next_id = np.random.choice(nucleus, p=nuc_probs)
        
        log_probs.append(np.log(probs[next_id] + 1e-10))
        context.append(next_id)
        
        if id2tok[next_id] == "<eos>":
            break
    
    tokens = [id2tok[i] for i in context]
    total_logp = sum(log_probs)
    avg_logp = total_logp / len(log_probs) if log_probs else 0
    ppl = np.exp(-avg_logp)
    
    return tokens, log_probs, total_logp, ppl

# --- Generate with different strategies ---
print("=== Full Autoregressive Generation ===\n")
print(f"Vocabulary: {vocab_tiny}")
print(f"Start token: <bos>\n")

strategies = [
    ("greedy",  {"strategy": "greedy", "temperature": 1.0}),
    ("τ=0.5",   {"strategy": "sample", "temperature": 0.5}),
    ("τ=1.0",   {"strategy": "sample", "temperature": 1.0}),
    ("τ=2.0",   {"strategy": "sample", "temperature": 2.0}),
    ("top-k=3", {"strategy": "top_k", "temperature": 1.0, "top_k": 3}),
    ("top-p=0.9", {"strategy": "top_p", "temperature": 1.0, "top_p": 0.9}),
]

print(f"{'Strategy':<14} {'Generated sequence':<45} {'log P(seq)':<12} {'PPL':<8}")
print("-" * 82)

all_results = []
for name, kwargs in strategies:
    np.random.seed(42)
    tokens, logps, total_logp, ppl = generate_sequence("<bos>", **kwargs)
    seq_str = " ".join(tokens[1:])  # skip <bos>
    print(f"{name:<14} {seq_str:<45} {total_logp:<12.4f} {ppl:<8.2f}")
    all_results.append((name, tokens, logps, total_logp, ppl))

# --- Step-by-step trace for greedy ---
print("\n=== Step-by-Step Trace (Greedy) ===\n")
context = [tok2id["<bos>"]]
print(f"{'Step':<6} {'Context':<25} {'Top-3 next tokens':<35} {'Chosen':<10} {'P(chosen)':<10}")
print("-" * 90)

for step in range(7):
    logits = tiny_lm_logits(context)
    probs = softmax_stable(logits)
    
    top3_idx = np.argsort(probs)[::-1][:3]
    top3_str = ", ".join(f"{id2tok[i]}({probs[i]:.3f})" for i in top3_idx)
    
    chosen = np.argmax(probs)
    ctx_str = " ".join(id2tok[i] for i in context)
    print(f"{step:<6} {ctx_str:<25} {top3_str:<35} {id2tok[chosen]:<10} {probs[chosen]:<10.4f}")
    
    context.append(chosen)
    if id2tok[chosen] == "<eos>":
        break

# --- Probability decomposition via chain rule ---
print("\n=== Chain Rule Probability Decomposition ===\n")
greedy_tokens, greedy_logps, _, _ = all_results[0][1], all_results[0][2], None, None
greedy_tokens = all_results[0][1]
greedy_logps = all_results[0][2]

print("P(sequence) = P(x₁|<bos>) × P(x₂|x₁) × ... × P(xₜ|x_{<t})")
print()
running_logp = 0.0
for i, (tok, lp) in enumerate(zip(greedy_tokens[1:], greedy_logps)):
    running_logp += lp
    print(f"  Step {i+1}: P({tok:>5} | {' '.join(greedy_tokens[:i+1]):<20}) = e^{lp:.4f} = {np.exp(lp):.4f}  |  cumulative log P = {running_logp:.4f}")

print(f"\n  Total: log P(sequence) = {running_logp:.4f}")
print(f"  P(sequence) = {np.exp(running_logp):.6f}")
print(f"  Perplexity = exp(-avg log P) = {np.exp(-running_logp/len(greedy_logps)):.2f}")

## §12 Summary — Language Model Probability Mathematics

| § | Topic | Key Formula | ML Application |
|---|-------|-------------|----------------|
| 1 | Softmax & Log-Softmax | $p_i = e^{z_i}/\sum e^{z_j}$ | Convert logits → probabilities |
| 2 | Cross-Entropy Loss | $\mathcal{L} = -\log p_k$, $\nabla_{z_i}\mathcal{L} = p_i - \mathbf{1}[i=k]$ | Training objective + clean gradient |
| 3 | Information Theory | $H, H(P,Q), D_{\text{KL}}, \text{PPL}$ | Model evaluation and comparison |
| 4 | Temperature Scaling | $p_i = \text{softmax}(z_i / \tau)$ | Control randomness in generation |
| 5 | Decoding Strategies | Greedy, top-k, top-p, min-p, typical | Trade off quality vs diversity |
| 6 | Beam Search | Score = $\log P / \text{len}^\alpha$ | Structured search for best sequence |
| 7 | Scaling Laws | $L(N) \propto N^{-0.076}$, Chinchilla 20:1 | Optimal compute allocation |
| 8 | DPO / Alignment | $\mathcal{L} = -\log\sigma(\beta \Delta)$ | Align models to human preferences |
| 9 | Calibration | ECE, temperature scaling | Trustworthy confidence estimates |
| 10 | Label Smoothing | $y_i = (1-\epsilon)\delta_{ik} + \epsilon/V$ | Prevent overconfidence, improve generalisation |
| 11 | Generation Pipeline | Chain rule: $P(x) = \prod P(x_t \mid x_{<t})$ | End-to-end autoregressive generation |

### Key Takeaways

1. **Softmax converts logits to probabilities** — always use the numerically stable version (subtract max)
2. **Cross-entropy gradient is elegant** — just `predicted - true`, enabling efficient backpropagation
3. **Perplexity = exp(cross-entropy)** — the universal evaluation metric (lower = better)
4. **Temperature controls the entropy-quality trade-off** — τ→0 is greedy, τ→∞ is uniform
5. **Decoding ≠ training** — sampling strategies shape generation without changing model weights
6. **Scaling laws are power laws** — more compute helps, but allocation (N vs D) matters enormously
7. **DPO eliminates the reward model** — direct preference optimisation via log-probability ratios
8. **Calibration is often overlooked** — overconfident models need post-hoc temperature scaling

---
*Next: [Training at Scale](../06-Training-at-Scale/notes.md) — distributed training, gradient accumulation, mixed precision*